# Seminar 06: Variational Auto Encoders (VAE)

**Date**: 2025/02/18

## 1. Environment Setup

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('../utils/datasets'))

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
import celeba
from tqdm.auto import tqdm

# Set device
cuda = torch.cuda.is_available()
device = torch.device("cuda" if cuda else ("mps" if torch.backends.mps.is_available() else "cpu"))
torch.backends.cudnn.benchmark = cuda

print(f"Using device: {device}")

## 2. Data Loading

We will use the [CelebA dataset](https://paperswithcode.com/dataset/celeba), a large-scale dataset of celebrity faces.

In [ ]:
# Data Hyperparameters
image_size = 64
batch_size = 128

# Define transformations
data_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),  
    transforms.Normalize([0.5]*3, [0.5]*3),  # Normalize each channel
])

# Load dataset
dataset = celeba.CelebADataset(root_dir='../data/celeba', transform=data_transform)
dataloader = torch.utils.data.DataLoader(dataset=dataset, batch_size=batch_size, shuffle=True, drop_last=True, pin_memory=cuda)

# Display dataset images
def show_images(dataloader, num_images=25, batch_idx=0):
    """Displays a grid of images from the dataloader."""
    for i, (images, _) in enumerate(dataloader):
        if i == batch_idx:
            grid = make_grid(images[:num_images], nrow=5, normalize=True, value_range=(-1, 1))
            plt.figure(figsize=(8, 8))
            plt.imshow(grid.permute(1, 2, 0).cpu().numpy().clip(0, 1))
            plt.axis("off")
            plt.title(f"Sample CelebA Images - Batch {batch_idx}")
            plt.show()
            break

# Display sample images
show_images(dataloader)

## 3. Why Standard Autoencoders Fail?

**Problem:** Irregular Latent Space

A standard autoencoder encodes input $\mathbf{x}$ into a **fixed latent vector $\mathbf{z}$**. This approach has a major limitation:
- It learns a compressed representation.
- The latent space is often **discontinuous** or **poorly structured**.

![](https://miro.medium.com/max/1000/1*83S0T8IEJyudR_I5rI9now@2x.png)

**Solution:** Variational Autoencoders (VAEs)

Instead of mapping each input to a single point in latent space, VAEs learn **a probability distribution over latent representations**. This ensures:

- Each input $x$ is mapped to a **Gaussian distribution** in latent space.
- The latent space is **continuous**, allowing smooth interpolation.
- It is possible to **generate new data** by sampling from the learned latent space.

## 4. Variational Approximation and ELBO

### The Challenge of Computing $p(\mathbf{z} \mid \mathbf{x})$

Using Bayes’ Theorem:

$$
p(\mathbf{z} \mid \mathbf{x}) = \frac{p(\mathbf{x} \mid \mathbf{z}) p(\mathbf{z})}{p(\mathbf{x})}
$$

The denominator, $p(\mathbf{x})$, requires integrating over all possible latent variables:

$$
p(\mathbf{x}) = \int p(\mathbf{x} \mid \mathbf{z}) p(\mathbf{z}) d\mathbf{z}
$$

This integral is **intractable** for high-dimensional data, making exact inference impossible.

### Solution: Variational Inference
Instead of directly computing $p(\mathbf{z} \mid \mathbf{x})$, we approximate it with a learned distribution:

$$
q(\mathbf{z} \mid \mathbf{x}) = \mathcal{N}(\boldsymbol{\mu}, \operatorname{diag}(\boldsymbol{\sigma}^2))
$$

where:
- $\boldsymbol{\mu}(\mathbf{x})$ is the **mean vector** of the latent distribution.
- $\boldsymbol{\sigma}^2(\mathbf{x})$ is the **diagonal covariance** representing uncertainty.

By applying **Jensen’s Inequality**, we derive the **Evidence Lower Bound (ELBO)**:

$$
\log p(\mathbf{x}) \geq \mathbb{E}_{q(\mathbf{z} \mid \mathbf{x})} [\log p(\mathbf{x} \mid \mathbf{z})] - D_{KL}(q(\mathbf{z} \mid \mathbf{x}) \parallel p(\mathbf{z}))
$$

This provides a **lower bound** on the data log-likelihood, which we **maximize** during training.

### Final Objective Function
To train the VAE, we maximize the ELBO, which consists of two terms:

1. **Reconstruction Loss** $\mathbb{E}_{q(\mathbf{z} \mid \mathbf{x})} [\log p(\mathbf{x} \mid \mathbf{z})]$:
   - Encourages the decoder to generate realistic reconstructions.
   - Measures how well $p(\mathbf{x} \mid \mathbf{z})$ reconstructs the input $\mathbf{x}$.
   
2. **KL Divergence** $D_{KL}(q(\mathbf{z} \mid \mathbf{x}) \parallel p(\mathbf{z}))$:
   - Regularizes the latent space by forcing $q(\mathbf{z} \mid \mathbf{x})$ to be close to the prior $p(\mathbf{z})$.
   - Ensures a well-structured and continuous latent space.


| ![](https://lilianweng.github.io/posts/2018-08-12-vae/vae-gaussian.png) | 
|:--:| 
| *[Source](https://lilianweng.github.io/posts/2018-08-12-vae/)* |

## 5. Model Implementation

### Encoder Network

In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_channels, encoder_feature_maps, latent_dim, image_size):
        """
        Encoder Network for Variational Autoencoder (VAE).

        Parameters:
            input_channels (int): Number of input image channels (e.g., 3 for RGB).
            encoder_feature_maps (int): Number of feature maps in the encoder.
            latent_dim (int): Dimensionality of the latent space.
            image_size (int): Size of input images.
        """
        super(Encoder, self).__init__()

        # Compute the downsampled spatial size
        self.downsampled_size = image_size // 16

        # Convolutional layers for feature extraction
        self.encoder = nn.Sequential(
            nn.Conv2d(input_channels, encoder_feature_maps, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.BatchNorm2d(encoder_feature_maps),

            nn.Conv2d(encoder_feature_maps, encoder_feature_maps * 2, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.BatchNorm2d(encoder_feature_maps * 2),

            nn.Conv2d(encoder_feature_maps * 2, encoder_feature_maps * 4, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.BatchNorm2d(encoder_feature_maps * 4),

            nn.Conv2d(encoder_feature_maps * 4, encoder_feature_maps * 8, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
        )

        # Fully connected layers to map features to latent space (mean & log variance)
        self.mean_layer = nn.Linear(encoder_feature_maps * 8 * self.downsampled_size * self.downsampled_size, latent_dim)
        self.logvar_layer = nn.Linear(encoder_feature_maps * 8 * self.downsampled_size * self.downsampled_size, latent_dim)

    def forward(self, x):
        batch_size = x.size(0)

        # Encode input image into feature maps
        hidden = self.encoder(x).view(batch_size, -1)

        # Compute mean and log variance of latent space
        mean, logvar = self.mean_layer(hidden), self.logvar_layer(hidden)

        return mean, logvar

### Decoder Network

In [ ]:
class Decoder(nn.Module):
    def __init__(self, input_channels, decoder_feature_maps, latent_dim, image_size):
        """
        Decoder Network for Variational Autoencoder (VAE).

        Parameters:
            input_channels (int): Number of output image channels (e.g., 3 for RGB).
            decoder_feature_maps (int): Number of feature maps in the decoder.
            latent_dim (int): Dimensionality of the latent space.
            image_size (int): Size of generated images.
        """
        super(Decoder, self).__init__()

        self.decoder_feature_maps = decoder_feature_maps
        self.upsampled_size = image_size // 16

        # Fully connected layer to expand latent space to feature maps
        self.decoder_dense = nn.Sequential(
            nn.Linear(latent_dim, decoder_feature_maps * 8 * self.upsampled_size * self.upsampled_size),
            nn.ReLU(inplace=True)
        )

        # Deconvolutional layers to generate images
        self.decoder_conv = nn.Sequential(
            nn.Upsample(scale_factor=2),
            nn.Conv2d(decoder_feature_maps * 8, decoder_feature_maps * 4, kernel_size=3, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.BatchNorm2d(decoder_feature_maps * 4),

            nn.Upsample(scale_factor=2),
            nn.Conv2d(decoder_feature_maps * 4, decoder_feature_maps * 2, kernel_size=3, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.BatchNorm2d(decoder_feature_maps * 2),

            nn.Upsample(scale_factor=2),
            nn.Conv2d(decoder_feature_maps * 2, decoder_feature_maps, kernel_size=3, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.BatchNorm2d(decoder_feature_maps),

            nn.Upsample(scale_factor=2),
            nn.Conv2d(decoder_feature_maps, input_channels, kernel_size=3, padding=1),
            nn.Tanh()  # Normalize output to [-1,1] range
        )

    def forward(self, latent_vector):
        batch_size = latent_vector.size(0)

        # Expand latent vector to match feature maps
        hidden = self.decoder_dense(latent_vector).view(
            batch_size, self.decoder_feature_maps * 8, self.upsampled_size, self.upsampled_size
        )

        return self.decoder_conv(hidden)

### VAE Model

#### The Reparameterization Trick
- The encoder samples $\mathbf{z}$ from $q(\mathbf{z} \mid \mathbf{x}) = \mathcal{N}(\boldsymbol{\mu}, \text{diag}(\boldsymbol{\sigma}^2))$.
- However, direct sampling prevents backpropagation.

##### **Solution**
Instead of sampling directly, we reparameterize:

$$
\mathbf{z} = \boldsymbol{\mu} + \boldsymbol{\sigma} \odot \boldsymbol{\epsilon}, \quad \text{where} \quad \boldsymbol{\epsilon} \sim \mathcal{N}(\mathbf{0}, \mathbf{I})
$$

This allows gradients to flow through $\boldsymbol{\mu}$ and $\boldsymbol{\sigma}$ during training.

In [ ]:
class VAE(nn.Module):
    def __init__(self, input_channels=3, decoder_feature_maps=32, encoder_feature_maps=32, latent_dim=100, image_size=64):
        """
        Variational Autoencoder (VAE) Model.

        Parameters:
            input_channels (int): Number of image channels (3 for RGB).
            decoder_feature_maps (int): Number of feature maps in the decoder.
            encoder_feature_maps (int): Number of feature maps in the encoder.
            latent_dim (int): Dimensionality of the latent space.
            image_size (int): Input image size.
            device (torch.device, optional): Computation device (CPU/GPU/MPS).
        """
        super(VAE, self).__init__()

        self.latent_dim = latent_dim
        self.device = device

        # Initialize encoder and decoder
        self.encoder = Encoder(input_channels, encoder_feature_maps, latent_dim, image_size)
        self.decoder = Decoder(input_channels, decoder_feature_maps, latent_dim, image_size)

    def forward(self, x):
        """Encodes input x into latent space and reconstructs it."""
        mean, logvar = self.encoder(x)
        latent_z = self.reparametrize(mean, logvar)
        reconstructed_x = self.decode(latent_z)
        return reconstructed_x, mean, logvar

    def encode(self, x):
        """Encodes an input image x into the latent space."""
        mean, logvar = self.encoder(x)
        latent_z = self.reparametrize(mean, logvar)
        return latent_z, mean, logvar

    def decode(self, latent_vector):
        """Decodes a latent vector into an image."""
        return self.decoder(latent_vector)

    @staticmethod
    def reparametrize(mean, logvar):
        """
        Applies the reparameterization trick: z = μ + ϵ * σ.

        Parameters:
            mean (Tensor): Mean of the latent distribution.
            logvar (Tensor): Log variance of the latent distribution.

        Returns:
            Tensor: Sampled latent vector.
        """
        std_dev = torch.exp(0.5 * logvar)
        epsilon = torch.randn_like(std_dev)
        return mean + epsilon * std_dev

    def sample(self, num_samples):
        """
        Generates new images from random noise.

        Parameters:
            num_samples (int): Number of images to generate.

        Returns:
            Tensor: Generated images.
        """
        sample_z = torch.randn(num_samples, self.latent_dim).to(self.device)
        return self.decode(sample_z)

## 6. Training VAE

### VAE loss function

$$
\mathcal{L_{\text{VAE}}} = -\mathbb{E}_{q(z|x)}[\log p(x|z)] + D_{KL}(q(z|x) \parallel p(z))
$$

### Computing the KL Divergence

We assume:
- The **prior** follows a **standard normal distribution**:
  
  $$
  p(\mathbf{z}) = \mathcal{N}(\mathbf{0}, \mathbf{I})
  $$

- The **posterior** is a **diagonal Gaussian distribution**:

  $$
  q(\mathbf{z} \mid \mathbf{x}) = \mathcal{N}(\boldsymbol{\mu}, \boldsymbol{\Sigma}))
  $$

where:
- $\boldsymbol{\mu} \in \mathbb{R}^{d}$ is the **mean vector**.
- $\boldsymbol{\Sigma} = \text{diag}(\boldsymbol{\sigma}^2)$ is the **diagonal covariance matrix**.
- $d$ is the size of the latent space.

The **KL divergence**:

$$
D_{KL}(q(\mathbf{z} \mid \mathbf{x}) \parallel p(\mathbf{z})) = \frac{1}{2} \left( \operatorname{Tr}(\boldsymbol{\Sigma}) + \boldsymbol{\mu}^T \boldsymbol{\mu} - d - \log \det \boldsymbol{\Sigma} \right)
$$

Since $\boldsymbol{\Sigma}$ is **diagonal**, the determinant simplifies $\log \det \boldsymbol{\Sigma} = \sum_{i=1}^{d} \log \sigma_i^2$. Thus, the final **KL term** is:

$$
\begin{align*}
D_{KL} &= \frac{1}{2} \sum_{i=1}^{d} \left( \sigma_i^2 + \mu_i^2 - 1 - \log \sigma_i^2 \right) \\
&= \frac{1}{2} \left( \|\boldsymbol{\mu}\|_2^2 + \|\boldsymbol{\sigma}\|_2^2 - d - \|\log \boldsymbol{\sigma}\|_1 \right)
\end{align*}
$$

In [ ]:
def loss_function(recon_x, x, mean, logvar, beta=1.0):
    """
    Computes the VAE loss: Reconstruction Loss + KL Divergence.

    Parameters:
        recon_x (Tensor): Reconstructed images.
        x (Tensor): Original images.
        mean (Tensor): Mean of the latent distribution.
        logvar (Tensor): Log variance of the latent distribution.
        beta (float): KL divergence scaling factor (default: 1.0 for standard VAE).

    Returns:
        mse_loss (Tensor): Mean Squared Error loss (Reconstruction).
        kld_loss (Tensor): KL Divergence loss.
    """
    batch_size = recon_x.shape[0]

    # Mean Squared Error (Reconstruction Loss) - Normalized per batch
    mse_loss = F.mse_loss(recon_x.view(batch_size, -1), x.view(batch_size, -1), reduction='sum') / batch_size

    # KL Divergence Loss - Normalized per batch
    kld_loss = -0.5 * torch.sum(1 + logvar - mean.pow(2) - torch.clamp(logvar.exp(), min=1e-8)) / batch_size

    # Apply beta scaling (for β-VAE)
    kld_loss *= beta

    return mse_loss, kld_loss

In [ ]:
def train_vae(model, dataloader, optimizer, scheduler, epochs, device, log_interval=50):
    """
    Trains a Variational Autoencoder (VAE) model.

    Parameters:
        model (nn.Module): The VAE model.
        dataloader (torch.utils.data.DataLoader): Dataloader for training.
        optimizer (torch.optim.Optimizer): Optimizer for training.
        scheduler (torch.optim.lr_scheduler._LRScheduler): Learning rate scheduler.
        epochs (int): Number of training epochs.
        device (torch.device): Computation device (CPU/GPU/MPS).
        log_interval (int): Number of batches between progress updates (default: 50).

    Returns:
        dict: Dictionary containing lists of recorded loss values over training.
    """
    model.to(device)
    model.train()

     # Initialize loss tracking
    loss_history = {
        "total_loss": [],
        "mse_loss": [],
        "kl_loss": [],
        "steps": []
    }

    with tqdm(total=epochs * len(dataloader), desc="Training Progress") as pbar:
        for epoch in range(1, epochs + 1):
            epoch_loss, epoch_mse, epoch_kld = 0, 0, 0

            for batch_idx, (data, _) in enumerate(dataloader):
                data = data.to(device)
                optimizer.zero_grad()

                # Forward pass
                recon_batch, mean, logvar = model(data)

                # Compute losses
                mse_loss, kld_loss = loss_function(recon_batch, data, mean, logvar)
                loss = mse_loss + kld_loss

                # Backpropagation
                loss.backward()
                optimizer.step()

                # Track losses
                epoch_loss += loss.item()
                epoch_mse += mse_loss.item()
                epoch_kld += kld_loss.item()

                # Save loss values at log intervals
                if batch_idx % log_interval == 0:
                    loss_history["total_loss"].append(loss.item() / len(data))
                    loss_history["mse_loss"].append(mse_loss.item() / len(data))
                    loss_history["kl_loss"].append(kld_loss.item() / len(data))
                    loss_history["steps"].append(epoch + batch_idx / len(dataloader))

                    pbar.set_description(
                        f"Epoch [{epoch}/{epochs}] | KL: {kld_loss.item()/len(data):.4f} | MSE: {mse_loss.item()/len(data):.4f}"
                    )

                # Update progress bar every `log_interval` steps
                if batch_idx % log_interval == 0:
                    pbar.set_description(
                        f"Epoch [{epoch}/{epochs}] | KL: {kld_loss.item()/len(data):.4f} | MSE: {mse_loss.item()/len(data):.4f}"
                    )
                pbar.update(1)

            # Compute average losses per epoch
            avg_loss = epoch_loss / len(dataloader.dataset)
            avg_mse = epoch_mse / len(dataloader.dataset)
            avg_kld = epoch_kld / len(dataloader.dataset)

            print(f"\nEpoch {epoch}/{epochs} | Avg Loss: {avg_loss:.4f} | Avg MSE: {avg_mse:.4f} | Avg KL: {avg_kld:.4f}")

            # Adjust learning rate
            scheduler.step()

            # Generate and visualize sample images after each epoch
            with torch.no_grad():
                sample_images = model.sample(64).cpu()
                image_grid = make_grid(sample_images, normalize=True)

                plt.figure(figsize=(8, 8))
                plt.imshow(image_grid.permute(1, 2, 0))
                plt.axis("off")
                plt.title(f"Generated Samples (Epoch {epoch})")
                plt.show()

    return loss_history

In [ ]:
# Training Hyperparameters
epochs = 10
lr = 1e-3

# Initialize Model, Optimizer & Scheduler
model = VAE().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

# Train the VAE
loss_history = train_vae(model, dataloader, optimizer, scheduler, epochs=epochs, device=device)

In [ ]:
def plot_loss(loss_history, use_log_scale=False):
    """
    Plots the training loss components as three subplots.

    Parameters:
        loss_history (dict): Dictionary containing tracked loss values.
        use_log_scale (bool): Whether to use log scale for loss values (default: False).

    Returns:
        None
    """
    _, axes = plt.subplots(3, 1, figsize=(10, 12), sharex=True)

    # Titles for subplots
    loss_titles = ["Total Loss", "Reconstruction Loss (MSE)", "KL Divergence"]
    loss_keys = ["total_loss", "mse_loss", "kl_loss"]
    colors = ["blue", "green", "red"]

    # Plot each loss component
    for i, ax in enumerate(axes):
        ax.plot(loss_history["steps"], loss_history[loss_keys[i]], label=loss_titles[i], color=colors[i], lw=2, alpha=0.8)
        
        # Set labels and title for each subplot
        ax.set_ylabel("Loss", fontsize=12)
        ax.set_title(loss_titles[i], fontsize=14)

        # Optionally apply log scale
        if use_log_scale:
            ax.set_yscale("log")

        ax.grid(True, linestyle="--", alpha=0.6)
        ax.legend(fontsize=10, loc="upper right")

    # Set x-axis label only on the last subplot
    axes[-1].set_xlabel("Training Step", fontsize=12)

    # Adjust layout
    plt.tight_layout()
    plt.show()

In [ ]:
plot_loss(loss_history)

## 7. Generating Images

In [ ]:
with torch.no_grad():
    generated_images = model.sample(64).cpu()
    grid = make_grid(generated_images, nrow=8, normalize=True)
    
    plt.figure(figsize=(8, 8))
    plt.imshow(grid.permute(1, 2, 0))
    plt.axis("off")
    plt.title("Generated Images")
    plt.show()

## 8. β-VAE: Controlling Disentanglement

$$
\mathcal{L_{\beta\text{-VAE}}} = -\mathbb{E}_{q(z|x)}[\log p(x|z)] + \beta D_{KL}(q(z|x) \parallel p(z))
$$

- $\beta = 1$ → Standard VAE with balanced reconstruction and regularization.  
- $\beta > 1$ → **More disentangled latent space**, enforcing independence among features.  
- $\beta < 1$ → **More emphasis on reconstruction**, leading to a less regularized latent space.


### 8.1 **Why Introduce $\beta$ (Beta-VAE)?**:
   - **Trade-off Control**: In the standard VAE, the reconstruction and regularization (KL divergence) are balanced (with $\beta=1$). However, sometimes this balance doesn't lead to latent representations that are easily interpretable.
   - **Enhanced Regularization**: By introducing a parameter $\beta$ that multiplies the KL divergence term, we can control how strongly the model is forced to conform to the prior.
     - $\beta > 1$: Increases the weight of the KL term, forcing the latent space to adhere more strictly to the prior distribution. This tends to encourage the dimensions to capture independent factors of variation.
     - $\beta < 1$: Puts more emphasis on reconstruction accuracy at the expense of latent regularization, which might lead to a more entangled latent space.

### 8.2 **What Does "Disentangled" Mean?**:
   - **Definition**: A disentangled representation is one in which individual latent variables are sensitive to changes in a single generative factor of the data, while being invariant to others.
   - **Practical Example**: In a dataset of celebrity faces, one latent dimension might capture "smile" while another controls "head orientation". Changing one dimension should ideally affect only one interpretable aspect of the image.
   - **Benefits**: Such representations make it easier to:
     - Understand and manipulate specific features (e.g., adjusting lighting or pose independently).
     - Transfer learning tasks where understanding independent factors is crucial.
     - Enhance interpretability and robustness in downstream applications.
A **β-VAE** introduces a weighting term $\beta$ on the KL divergence:



